# Phase 3 — Task 3.1: PCA-Based Statistical Risk Model

**Objective:** Construct a rolling covariance matrix $\Sigma(t)$ for the full S&P 500 universe using a PCA-based factor model, following the professor's simplified approach:

1. **Stratified core selection:** ~55 stocks from 11 GICS sectors via intra-sector market-cap stratification
2. **Core covariance:** Sample covariance from up to 10 years of daily data (no pre-PCA shrinkage)
3. **PCA:** Extract 3–15 statistical principal components (~90% variance explained)
4. **Universe regression:** Time-series OLS of all ~600 stocks on PC returns → factor loadings β and idiosyncratic variance Λ\_ε
5. **Reconstruct:** $\Sigma = \beta\,\Omega_f\,\beta' + \Lambda_\varepsilon$ (monthly frequency)

---

**Inputs:**
| File | Description |
|---|---|
| `sp500_price_panel_with_delist.csv.gz` | Daily returns (2008–2024) |
| `sp500_expected_returns.csv.gz` | Task 2.4 output — defines monthly universe |

**Outputs:**
| File | Description |
|---|---|
| `sp500_risk_model.pkl` | Monthly risk model components {β, Ω_f, Λ\_ε, permnos} |

In [1]:
# ============================================================
# Cell 1: Imports & Load Data
# ============================================================
import pandas as pd
import numpy as np
import pickle
import os
import time
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = "data/"

# --- Daily price panel ---
print("Loading daily price panel...")
t0 = time.time()
parquet_path = DATA_DIR + "parquet_backup/sp500_price_panel_with_delist.parquet"
csv_path = DATA_DIR + "sp500_price_panel_with_delist.csv.gz"

if os.path.exists(parquet_path):
    daily = pd.read_parquet(parquet_path,
                            columns=["permno", "date", "ret_adj"])
    print(f"  Loaded parquet: {daily.shape[0]:,} rows ({time.time()-t0:.1f}s)")
else:
    daily = pd.read_csv(csv_path, compression="gzip",
                        usecols=["permno", "date", "ret_adj"])
    print(f"  Loaded CSV: {daily.shape[0]:,} rows ({time.time()-t0:.1f}s)")

daily["date"] = pd.to_datetime(daily["date"])

# --- Expected returns (defines universe per month) ---
er = pd.read_csv(DATA_DIR + "sp500_expected_returns.csv.gz",
                 compression="gzip")
er["date"] = pd.to_datetime(er["date"])
er["month"] = er["date"].dt.to_period("M")

print(f"\nExpected returns: {er.shape[0]:,} rows, {er['month'].nunique()} months")
print(f"Period: {er['month'].min()} → {er['month'].max()}")
print(f"Avg stocks/month: {er.groupby('month')['permno'].nunique().mean():.0f}")

Loading daily price panel...
  Loaded parquet: 2,934,733 rows (0.1s)

Expected returns: 99,363 rows, 157 months
Period: 2011-12 → 2024-12
Avg stocks/month: 633


## Data Preprocessing: Pivot to Wide Format

Convert the long-format daily panel to a (dates × stocks) matrix for fast slicing. This is the core data structure for all subsequent computations.

In [2]:
# ============================================================
# Cell 2: Pivot Daily Returns to Wide Format
# ============================================================
print("Pivoting to wide format...")
t0 = time.time()

# Verify uniqueness
assert not daily.duplicated(["date", "permno"]).any(), \
    "Duplicate (date, permno) in daily panel!"

daily_wide = daily.pivot(index="date", columns="permno", values="ret_adj")
daily_wide.sort_index(inplace=True)

print(f"  Shape: {daily_wide.shape[0]:,} dates × {daily_wide.shape[1]:,} stocks "
      f"({time.time()-t0:.1f}s)")
print(f"  Date range: {daily_wide.index[0].date()} → {daily_wide.index[-1].date()}")
print(f"  NaN rate: {daily_wide.isna().mean().mean():.2%}")

# Free raw daily DataFrame
del daily

# Quick check: all universe permnos should exist in daily_wide
all_er_permnos = set(er["permno"].unique())
in_daily = set(daily_wide.columns)
missing = all_er_permnos - in_daily
if missing:
    print(f"  Warning: {len(missing)} permnos in expected_returns not in daily panel")
else:
    print(f"  All {len(all_er_permnos)} universe permnos found in daily panel")

Pivoting to wide format...
  Shape: 4,279 dates × 871 stocks (0.9s)
  Date range: 2008-01-02 → 2024-12-31
  NaN rate: 21.39%
  All 793 universe permnos found in daily panel


## Stratified Core Stock Selection

To avoid large-cap bias in the PCA factors, we use **intra-sector stratified sampling**:

1. Within each GICS sector, sort stocks by market cap
2. Divide into $n$ equal-count strata (market-cap buckets)
3. From each stratum, select the stock closest to the stratum's median market cap
4. Only consider stocks with ≥200 daily observations in the past year

This ensures the core set spans the full size spectrum within each sector — from "relative small-caps" to mega-caps — giving the PCA a representative cross-section to extract risk factors from.

In [3]:
# ============================================================
# Cell 3: Stratified Core Stock Selection
# ============================================================

N_PER_SECTOR = 5          # 5 per sector × 11 sectors ≈ 55 core stocks
MIN_DAILY_OBS = 200

def stratified_select_core(universe_df, daily_obs_counts,
                           n_per_sector=N_PER_SECTOR,
                           min_obs=MIN_DAILY_OBS):
    '''
    Stratified sampling within GICS sectors by market cap.

    Within each sector, divide stocks into n_per_sector market-cap strata
    (equal count). From each stratum, pick the stock nearest to the
    stratum median market cap. Only stocks with sufficient daily data
    are eligible.

    Parameters
    ----------
    universe_df : DataFrame with [permno, gsector, mkt_cap]
    daily_obs_counts : Series, permno -> count of non-NaN daily obs
    n_per_sector : int, strata per sector (= stocks per sector)
    min_obs : int, minimum daily observations required

    Returns
    -------
    DataFrame of selected core stocks
    '''
    # Filter for data quality
    eligible_permnos = daily_obs_counts[daily_obs_counts >= min_obs].index
    eligible = universe_df[universe_df["permno"].isin(eligible_permnos)].copy()

    core = []
    for gsector, grp in eligible.groupby("gsector"):
        grp_s = grp.sort_values("mkt_cap").reset_index(drop=True)
        n = len(grp_s)
        n_strata = min(n_per_sector, n)
        if n_strata == 0:
            continue

        # Equal-count strata via rank partitioning
        boundaries = np.linspace(0, n, n_strata + 1, dtype=int)

        for i in range(n_strata):
            stratum = grp_s.iloc[boundaries[i]:boundaries[i+1]]
            if len(stratum) == 0:
                continue
            median_cap = stratum["mkt_cap"].median()
            best_idx = (stratum["mkt_cap"] - median_cap).abs().idxmin()
            selected = grp_s.loc[best_idx].copy()
            selected["stratum"] = i    # 0=smallest, n_strata-1=largest
            core.append(selected)

    return pd.DataFrame(core).reset_index(drop=True)


# --- Demo for the first rebalancing month ---
demo_month = er["month"].min()
demo_date = er.loc[er["month"] == demo_month, "date"].iloc[0]
demo_univ = er[er["month"] == demo_month][["permno", "gsector", "mkt_cap"]].copy()

one_yr_ago = demo_date - pd.DateOffset(years=1)
demo_counts = daily_wide.loc[one_yr_ago:demo_date].notna().sum()

demo_core = stratified_select_core(demo_univ, demo_counts)

print(f"Demo: {demo_month} (rebal date: {demo_date.date()})")
print(f"  Universe: {len(demo_univ)} stocks, {demo_univ['gsector'].nunique()} sectors")
print(f"  Core selected: {len(demo_core)} stocks, "
      f"{demo_core['gsector'].nunique()} sectors")

# Sector breakdown
print(f"\n  Sector breakdown:")
print(f"  {'Sector':>8s} {'N_core':>7s} {'Strata':>7s}  "
      f"{'MinCap($B)':>11s} {'MedCap($B)':>11s} {'MaxCap($B)':>11s}")
print("  " + "-" * 60)
for gs, sg in demo_core.groupby("gsector"):
    mn = sg["mkt_cap"].min() / 1e9
    md_val = sg["mkt_cap"].median() / 1e9
    mx = sg["mkt_cap"].max() / 1e9
    print(f"  {int(gs):>8d} {len(sg):>7d} {int(sg['stratum'].max())+1:>7d}"
          f"  {mn:>11.1f} {md_val:>11.1f} {mx:>11.1f}")

# Compare core vs universe size distribution
print(f"\n  Market cap distribution (log10 USD):")
print(f"  {'Percentile':>12s} {'Universe':>10s} {'Core':>10s} {'Diff':>8s}")
print("  " + "-" * 42)
for p in [10, 25, 50, 75, 90]:
    u = np.log10(demo_univ["mkt_cap"].quantile(p / 100))
    c = np.log10(demo_core["mkt_cap"].quantile(p / 100))
    print(f"  P{p:<11d} {u:>10.2f} {c:>10.2f} {c-u:>+8.2f}")

print(f"\n  Stratified sampling captures a wider market-cap range")
print(f"  than a pure top-N selection would.")

Demo: 2011-12 (rebal date: 2011-12-30)
  Universe: 678 stocks, 11 sectors
  Core selected: 55 stocks, 11 sectors

  Sector breakdown:
    Sector  N_core  Strata   MinCap($B)  MedCap($B)  MaxCap($B)
  ------------------------------------------------------------
        10       5       5          4.5        11.0        38.0
        15       5       5          2.9         6.1        32.0
        20       5       5          1.9         7.5        31.3
        25       5       5          2.1         5.4        31.7
        30       5       5          6.2        11.2       103.7
        35       5       5          1.4         7.9        48.1
        40       5       5          2.3         6.8        44.1
        45       5       5          1.5         6.9        34.5
        50       5       5          1.2         5.9       113.6
        55       5       5          4.1         8.6        25.7
        60       5       5          2.8         5.8        16.9

  Market cap distribution (log10 U

## Risk Model Pipeline Functions

Four core functions that execute for each rebalancing month:

1. `compute_core_covariance()` — **EWMA**-weighted covariance (half-life ≈ 2yr) + per-stock std dev
2. `extract_pca()` — converts EWMA covariance to **correlation** matrix, eigendecomposition, select top PCs
3. `regularize_risk_model()` — post-PCA shrinkage: blend full Σ toward its diagonal (α = 5%)
4. `run_regressions()` — OLS each stock on PC returns → β, σ²\_ε

In [12]:
# ============================================================
# Cell 4: Define Risk Model Pipeline Functions
# ============================================================

# --- Configuration ---
COV_MAX_YEARS  = 10       # max lookback for core covariance
COV_MIN_OBS    = 504      # min daily obs for covariance (~2 years)
EWMA_HALFLIFE  = 480      # EWMA half-life in trading days (~2 years)
REG_DAYS       = 252      # regression window (~1 year)
REG_MIN_OBS    = 120      # min valid obs per stock regression
N_PCS_MIN      = 3        # minimum PCs to retain
N_PCS_MAX      = 30       # maximum PCs to retain
VAR_TARGET     = 0.90     # cumulative variance target
MONTHLY_SCALE  = 21       # daily -> monthly (trading days)
FINAL_SHRINK   = 0.05     # post-PCA shrink toward diagonal (0 = off)


# ------------------------------------------------------------------ #
# 1. EWMA-weighted covariance
# ------------------------------------------------------------------ #
def compute_core_covariance(R_core):
    """
    EWMA-weighted covariance of core stock daily returns.

    Exponential weighting gives more weight to recent observations,
    producing a more responsive and timely covariance estimate while
    still using the full lookback window.

    Parameters
    ----------
    R_core : (T, p) ndarray – complete (no NaN) daily returns,
             rows in chronological order (earliest first).

    Returns
    -------
    cov       : (p, p) EWMA covariance matrix
    std_devs  : (p,)   per-stock daily std dev (sqrt of diag(cov))
    cond      : float  condition number of cov (max / min eigenvalue)
    """
    T, p = R_core.shape

    # EWMA weights: oldest row → decay^(T-1), newest row → decay^0 = 1
    decay = 0.5 ** (1.0 / EWMA_HALFLIFE)
    raw_w = decay ** np.arange(T - 1, -1, -1, dtype=np.float64)
    w = raw_w / raw_w.sum()

    # Weighted mean & centred returns
    mu  = w @ R_core                          # (p,)
    R_c = R_core - mu

    # Weighted covariance
    cov = (R_c * w[:, None]).T @ R_c          # (p, p)

    # Bessel-like correction using effective sample size
    n_eff = 1.0 / (w ** 2).sum()
    cov  *= n_eff / (n_eff - 1)

    std_devs = np.sqrt(np.diag(cov))

    eigvals = np.linalg.eigvalsh(cov)
    cond = eigvals[-1] / max(eigvals[0], 1e-20)

    return cov, std_devs, cond


# ------------------------------------------------------------------ #
# 2. PCA on correlation matrix
# ------------------------------------------------------------------ #
def extract_pca(cov, std_devs):
    """
    PCA on the CORRELATION matrix derived from the EWMA covariance.

    Why correlation?  The covariance matrix is dominated by high-vol
    stocks, spreading variance across many PCs.  Normalising to
    correlation captures pure co-movement structure (market, sector,
    style) which is inherently lower-dimensional.

    The returned eigenvectors are pre-multiplied by D^{-1} (diagonal
    of 1/σ_i) so that:
        PC_returns = raw_returns @ V_K     (no manual standardisation)

    Eigenvalues are in correlation space (sum = p).

    Returns
    -------
    V_K      : (p, K) transformed eigenvectors
    eig_K    : (K,)   eigenvalues (correlation space)
    n_pcs    : int    number of PCs selected
    cum_var  : float  cumulative variance explained (0-1)
    """
    D_inv = 1.0 / np.maximum(std_devs, 1e-20)
    corr  = cov * np.outer(D_inv, D_inv)

    eigenvalues, eigenvectors = np.linalg.eigh(corr)

    # Sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Select number of PCs
    cum_var = np.cumsum(eigenvalues) / eigenvalues.sum()
    n_pcs   = int(np.searchsorted(cum_var, VAR_TARGET) + 1)
    n_pcs   = max(N_PCS_MIN, min(N_PCS_MAX, n_pcs))

    V_K   = eigenvectors[:, :n_pcs]
    eig_K = eigenvalues[:n_pcs]

    # Pre-multiply by D^{-1} so PC_ret = R_raw @ V_transformed
    V_K_transformed = np.diag(D_inv) @ V_K

    return V_K_transformed, eig_K, n_pcs, float(cum_var[n_pcs - 1])


# ------------------------------------------------------------------ #
# 3. Post-PCA regularisation
# ------------------------------------------------------------------ #
def regularize_risk_model(beta, omega_f, lambda_eps, alpha=FINAL_SHRINK):
    """
    Shrink the full factor-model covariance toward its diagonal.

    Mathematically:
        Σ_reg = (1-α) Σ_raw + α diag(Σ_raw)
    where Σ_raw = β Ω_f β' + diag(Λ_ε).

    This is equivalent to:
        Ω_f_adj   = (1-α) · Ω_f
        Λ_ε_adj_i = Λ_ε_i + α · Σ_k β_{ik}² ω_k

    Per-stock total variance is preserved; off-diagonal (factor) 
    contribution is damped.  Improves conditioning for downstream
    portfolio optimisation without distorting relative risk.

    Returns: omega_f_adj (K,K), lambda_eps_adj (N,)
    """
    if alpha <= 0:
        return omega_f, lambda_eps

    factor_var      = np.sum(beta ** 2 * np.diag(omega_f), axis=1)   # (N,)
    omega_f_adj     = (1 - alpha) * omega_f
    lambda_eps_adj  = lambda_eps + alpha * factor_var

    return omega_f_adj, lambda_eps_adj


# ------------------------------------------------------------------ #
# 4. Universe regressions
# ------------------------------------------------------------------ #
def run_regressions(R_univ, PC_ret, core_valid_mask):
    """
    OLS each stock on PC returns:  r_i = α + β_i · f + ε_i

    Parameters
    ----------
    R_univ          : (T, N) ndarray, may contain NaN
    PC_ret          : (T, K) ndarray, valid on core_valid_mask dates
    core_valid_mask : (T,) bool

    Returns
    -------
    betas     : (N, K) factor loadings
    lambda_eps: (N,)   idiosyncratic variance (daily)
    reg_valid : (N,)   bool – True if regression succeeded
    """
    T, N = R_univ.shape
    K = PC_ret.shape[1]

    betas      = np.full((N, K), np.nan)
    lambda_eps = np.full(N, np.nan)
    reg_valid  = np.zeros(N, dtype=bool)

    for j in range(N):
        y  = R_univ[:, j]
        ok = ~np.isnan(y) & core_valid_mask
        n_ok = ok.sum()
        if n_ok < REG_MIN_OBS:
            continue

        y_v = y[ok]
        X   = np.column_stack([np.ones(n_ok), PC_ret[ok]])

        coeffs, _, _, _ = np.linalg.lstsq(X, y_v, rcond=None)
        betas[j]      = coeffs[1:]
        resid          = y_v - X @ coeffs
        lambda_eps[j]  = np.var(resid, ddof=X.shape[1])
        reg_valid[j]   = True

    return betas, lambda_eps, reg_valid


print("Pipeline functions defined.")
print(f"  Covariance : EWMA (half-life={EWMA_HALFLIFE}d ≈ "
      f"{EWMA_HALFLIFE/252:.1f}yr), up to {COV_MAX_YEARS}yr lookback")
print(f"  PCA        : on CORRELATION matrix, {N_PCS_MIN}–{N_PCS_MAX} PCs, "
      f"target {VAR_TARGET:.0%} var explained")
print(f"  Post-PCA   : shrink toward diagonal α={FINAL_SHRINK:.0%}")
print(f"  Regression : ~{REG_DAYS}d window, min {REG_MIN_OBS} obs")
print(f"  Scaling    : daily × {MONTHLY_SCALE} → monthly")

Pipeline functions defined.
  Covariance : EWMA (half-life=480d ≈ 1.9yr), up to 10yr lookback
  PCA        : on CORRELATION matrix, 3–30 PCs, target 90% var explained
  Post-PCA   : shrink toward diagonal α=5%
  Regression : ~252d window, min 120 obs
  Scaling    : daily × 21 → monthly


## Main Rolling Loop

For each of the 157 rebalancing months:
1. Identify universe stocks (from expected returns)
2. Stratified core selection (~55 stocks)
3. **EWMA** covariance (up to 10yr daily, half-life ≈ 2yr) + per-stock σ
4. PCA on **correlation** matrix → top 3–20 PCs (targeting 90% var explained)
5. Compute PC return series (1yr window)
6. Regress all universe stocks on PCs → β, σ²\_ε
7. **Post-PCA regularisation**: shrink Σ toward diagonal (α = 5%)
8. Scale to monthly frequency and store

In [13]:
# ============================================================
# Cell 5: Rolling Risk Model Estimation
# ============================================================

CORE_COV_COVERAGE = 0.80   # core stock must have >=80% non-NaN in cov window

print("=" * 70)
print("  ROLLING RISK MODEL ESTIMATION")
print("=" * 70)

rebal_months = sorted(er["month"].unique())
month_to_date = er.groupby("month")["date"].first().to_dict()

print(f"  Months: {len(rebal_months)} ({rebal_months[0]} -> {rebal_months[-1]})")

risk_models = {}
diag_records = []
t_start = time.time()

for m_idx, month in enumerate(rebal_months):
    rebal_date = month_to_date[month]

    # --- 1. Universe ---
    er_m = er[er["month"] == month]
    univ_permnos_raw = er_m["permno"].unique()
    # Intersect with daily_wide columns
    univ_permnos = np.array([p for p in univ_permnos_raw
                             if p in daily_wide.columns])
    N_univ = len(univ_permnos)

    # --- 2. Data availability in past year ---
    one_yr_ago = rebal_date - pd.DateOffset(years=1)
    reg_slice = daily_wide.loc[one_yr_ago:rebal_date]
    avail_cols = [p for p in univ_permnos if p in reg_slice.columns]
    daily_obs = reg_slice[avail_cols].notna().sum()

    # --- 3. Stratified core selection ---
    univ_df = er_m[er_m["permno"].isin(univ_permnos)][
        ["permno", "gsector", "mkt_cap"]
    ].drop_duplicates("permno")
    core_df = stratified_select_core(univ_df, daily_obs)
    core_permnos = core_df["permno"].values
    n_core = len(core_permnos)

    # --- 4. Core covariance (up to 10yr) ---
    # First, filter core stocks by data coverage in the cov window
    # to avoid dropna() discarding too many dates
    cov_start = rebal_date - pd.DateOffset(years=COV_MAX_YEARS)
    R_core_raw = daily_wide.loc[cov_start:rebal_date, core_permnos]
    T_cov_window = len(R_core_raw)

    if T_cov_window > 0:
        coverage = R_core_raw.notna().sum() / T_cov_window
        good_mask = coverage >= CORE_COV_COVERAGE
        core_permnos_cov = core_permnos[good_mask.values]
    else:
        core_permnos_cov = core_permnos

    # Need at least N_PCS_MAX + 1 core stocks for meaningful PCA
    if len(core_permnos_cov) < N_PCS_MAX + 1:
        if (m_idx + 1) % 24 == 0 or m_idx == 0:
            print(f"  {month}: SKIP (only {len(core_permnos_cov)} core stocks "
                  f"with sufficient coverage)")
        continue

    R_core_filtered = daily_wide.loc[cov_start:rebal_date, core_permnos_cov]
    R_core_clean = R_core_filtered.dropna().values   # complete cases
    n_cov_obs = R_core_clean.shape[0]

    if n_cov_obs < COV_MIN_OBS:
        if (m_idx + 1) % 24 == 0 or m_idx == 0:
            print(f"  {month}: SKIP (only {n_cov_obs} cov obs after filtering)")
        continue

    cov_mat, core_stds, cond_num = compute_core_covariance(R_core_clean)
    n_core_used = len(core_permnos_cov)

    # --- 5. PCA (on correlation matrix) ---
    V_K, eig_K, n_pcs, cum_var = extract_pca(cov_mat, core_stds)

    # --- 6. PC returns for regression window ---
    # Use the filtered core set for PCA projections
    R_core_reg = daily_wide.loc[one_yr_ago:rebal_date, core_permnos_cov].values
    T_reg = R_core_reg.shape[0]
    core_valid = ~np.isnan(R_core_reg).any(axis=1)

    PC_ret = np.full((T_reg, n_pcs), np.nan)
    if core_valid.sum() > 0:
        PC_ret[core_valid] = R_core_reg[core_valid] @ V_K

    # --- 7. Universe regressions ---
    R_univ = daily_wide.loc[one_yr_ago:rebal_date, univ_permnos].values
    betas_raw, lambda_raw, reg_valid = run_regressions(
        R_univ, PC_ret, core_valid
    )

    # --- 8. Assemble (monthly frequency) ---
    vi = np.where(reg_valid)[0]
    permnos_final = [int(univ_permnos[i]) for i in vi]
    beta_final = betas_raw[vi]                             # (N_valid, K)
    omega_f = np.diag(eig_K) * MONTHLY_SCALE              # (K, K)
    lambda_final = lambda_raw[vi] * MONTHLY_SCALE          # (N_valid,)

    # --- 9. Post-PCA regularisation ---
    omega_f, lambda_final = regularize_risk_model(
        beta_final, omega_f, lambda_final
    )

    risk_models[str(month)] = {
        "beta":            beta_final,
        "omega_f":         omega_f,
        "lambda_eps":      lambda_final,
        "permnos":         permnos_final,
        "n_pcs":           n_pcs,
        "cum_var_explained": cum_var,
        "core_permnos":    [int(p) for p in core_permnos_cov],
        "cond_number":     cond_num,
        "n_cov_days":      n_cov_obs,
    }

    # Diagnostics
    factor_var = np.sum(beta_final ** 2 * np.diag(omega_f), axis=1)
    total_var = factor_var + lambda_final

    diag_records.append({
        "month":          month,
        "n_universe":     N_univ,
        "n_core":         n_core,
        "n_core_used":    n_core_used,
        "n_sectors":      core_df["gsector"].nunique(),
        "n_cov_days":     n_cov_obs,
        "cond_number":    cond_num,
        "n_pcs":          n_pcs,
        "var_explained":  cum_var,
        "n_stocks_model": len(permnos_final),
        "pct_coverage":   len(permnos_final) / N_univ * 100,
        "factor_var_frac": (factor_var / total_var).mean(),
        "avg_ann_vol":    np.sqrt(total_var.mean() * 12),
    })

    # Progress: print every 12 months or first/last
    if (m_idx + 1) % 12 == 0 or m_idx == 0 or m_idx == len(rebal_months) - 1:
        elapsed = time.time() - t_start
        print(f"  {month}: core={n_core_used}/{n_core}, PCs={n_pcs}, "
              f"var_expl={cum_var:.1%}, "
              f"model={len(permnos_final)}/{N_univ}, "
              f"cond={cond_num:.0f}"
              f" ({elapsed:.0f}s)")

total_time = time.time() - t_start
print(f"\n  Completed: {len(risk_models)} months in {total_time:.1f}s "
      f"({total_time/max(len(risk_models),1):.2f}s/month)")

  ROLLING RISK MODEL ESTIMATION
  Months: 157 (2011-12 -> 2024-12)
  2011-12: core=52/55, PCs=29, var_expl=90.2%, model=678/678, cond=668 (0s)
  2012-11: core=53/55, PCs=30, var_expl=88.0%, model=676/676, cond=650 (3s)
  2013-11: core=51/55, PCs=30, var_expl=88.5%, model=684/684, cond=740 (7s)
  2014-11: core=50/55, PCs=30, var_expl=86.8%, model=681/681, cond=440 (10s)
  2015-11: core=51/55, PCs=30, var_expl=87.8%, model=667/668, cond=320 (13s)
  2016-11: core=50/55, PCs=30, var_expl=88.6%, model=647/647, cond=431 (16s)
  2017-11: core=51/55, PCs=30, var_expl=85.0%, model=644/644, cond=239 (19s)
  2018-11: core=51/55, PCs=30, var_expl=85.5%, model=622/622, cond=293 (23s)
  2019-11: core=51/55, PCs=30, var_expl=85.9%, model=616/616, cond=359 (26s)
  2020-11: core=49/55, PCs=28, var_expl=90.3%, model=606/606, cond=453 (29s)
  2021-11: core=52/55, PCs=30, var_expl=87.7%, model=601/601, cond=302 (32s)
  2022-11: core=53/55, PCs=30, var_expl=87.8%, model=580/580, cond=287 (35s)
  2023-11: c

In [14]:
# ============================================================
# Cell 6: Diagnostics Summary
# ============================================================

diag = pd.DataFrame(diag_records)

print("=" * 70)
print("  RISK MODEL DIAGNOSTICS")
print("=" * 70)

# Coverage
print(f"\n  Coverage:")
print(f"    Models produced: {len(diag)} months")
print(f"    Avg stocks/month: {diag['n_stocks_model'].mean():.0f}")
print(f"    Min stocks/month: {diag['n_stocks_model'].min()}")
print(f"    Avg coverage: {diag['pct_coverage'].mean():.1f}%")
print(f"    Min coverage: {diag['pct_coverage'].min():.1f}%")

# Core selection
print(f"\n  Core Stock Selection:")
print(f"    Avg core size: {diag['n_core'].mean():.0f}")
print(f"    Avg sectors covered: {diag['n_sectors'].mean():.1f}")

# EWMA + Correlation PCA
print(f"\n  EWMA Covariance + Correlation PCA:")
print(f"    Avg daily obs used: {diag['n_cov_days'].mean():.0f}")
print(f"    EWMA half-life: {EWMA_HALFLIFE}d (~{EWMA_HALFLIFE/252:.1f}yr)")
print(f"    Condition number (cov): mean={diag['cond_number'].mean():.0f}, "
      f"max={diag['cond_number'].max():.0f}")
print(f"    PCA on correlation matrix; post-PCA shrink α={FINAL_SHRINK:.0%}")

# PCA
print(f"\n  PCA:")
print(f"    PCs used: min={diag['n_pcs'].min()}, "
      f"max={diag['n_pcs'].max()}, "
      f"mode={diag['n_pcs'].mode().iloc[0]}")
print(f"    Cumulative variance explained: "
      f"mean={diag['var_explained'].mean():.1%}, "
      f"range=[{diag['var_explained'].min():.1%}, "
      f"{diag['var_explained'].max():.1%}]")

# Variance decomposition
print(f"\n  Variance Decomposition (post-regularisation):")
print(f"    Factor variance fraction: {diag['factor_var_frac'].mean():.1%} "
      f"(range: {diag['factor_var_frac'].min():.1%} - "
      f"{diag['factor_var_frac'].max():.1%})")
print(f"    Idiosyncratic fraction:   {1-diag['factor_var_frac'].mean():.1%}")
print(f"    Avg annualized vol:       {diag['avg_ann_vol'].mean():.1%}")

# Year-by-year evolution
diag["year"] = diag["month"].apply(lambda m: m.year)
print(f"\n  Year-by-Year Summary:")
print(f"  {'Year':>6s} {'N_model':>8s} {'N_PCs':>6s} {'VarExpl':>8s} "
      f"{'FactorFrac':>11s} {'AnnVol':>8s} {'Cond':>8s}")
print("  " + "-" * 62)
for year, yg in diag.groupby("year"):
    print(f"  {year:>6d} {yg['n_stocks_model'].mean():>8.0f} "
          f"{yg['n_pcs'].mean():>6.1f} {yg['var_explained'].mean():>8.1%} "
          f"{yg['factor_var_frac'].mean():>11.1%} "
          f"{yg['avg_ann_vol'].mean():>8.1%} "
          f"{yg['cond_number'].mean():>8.0f}")

  RISK MODEL DIAGNOSTICS

  Coverage:
    Models produced: 157 months
    Avg stocks/month: 633
    Min stocks/month: 579
    Avg coverage: 100.0%
    Min coverage: 99.8%

  Core Stock Selection:
    Avg core size: 55
    Avg sectors covered: 11.0

  EWMA Covariance + Correlation PCA:
    Avg daily obs used: 1956
    EWMA half-life: 480d (~1.9yr)
    Condition number (cov): mean=399, max=1097
    PCA on correlation matrix; post-PCA shrink α=5%

  PCA:
    PCs used: min=26, max=30, mode=30
    Cumulative variance explained: mean=88.0%, range=[83.5%, 90.7%]

  Variance Decomposition (post-regularisation):
    Factor variance fraction: 56.9% (range: 45.0% - 69.6%)
    Idiosyncratic fraction:   43.1%
    Avg annualized vol:       37.1%

  Year-by-Year Summary:
    Year  N_model  N_PCs  VarExpl  FactorFrac   AnnVol     Cond
  --------------------------------------------------------------
    2011      678   29.0    90.2%       69.6%    45.4%      668
    2012      674   30.0    88.8%       

## Validation

Key checks:
1. **β distribution:** Most loadings should be in [-3, 3]; market-like PC1 β should be near 1
2. **Model variance vs actual:** The risk model's predicted stock variance should correlate with realized trailing volatility
3. **Positive definiteness:** The factored covariance Σ = βΩβ' + Λ\_ε is guaranteed PD (diagonal Ω > 0, diagonal Λ > 0)

In [15]:
# ============================================================
# Cell 7: Validation
# ============================================================

print("=" * 70)
print("  VALIDATION")
print("=" * 70)

# --- 1. Beta distributions ---
# Collect betas from a representative month (midpoint)
# Use actual risk_models keys to avoid KeyError on skipped months
model_months = sorted(risk_models.keys())
mid_month = model_months[len(model_months) // 2]
rm_mid = risk_models[mid_month]
beta_mid = rm_mid["beta"]
n_pcs_mid = rm_mid["n_pcs"]

print(f"\n  Beta distributions (month: {mid_month}):")
print(f"  Stocks: {beta_mid.shape[0]}, PCs: {n_pcs_mid}")
print(f"  {'PC':>4s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s} "
      f"{'|beta|>2':>9s}")
print("  " + "-" * 46)
for k in range(n_pcs_mid):
    b = beta_mid[:, k]
    print(f"  PC{k+1:>2d} {b.mean():>8.4f} {b.std():>8.4f} "
          f"{b.min():>8.4f} {b.max():>8.4f} "
          f"{(np.abs(b) > 2).mean()*100:>8.1f}%")

print(f"\n  Note: PC1 typically captures market-wide risk;")
print(f"  its beta ~ 1 indicates stocks move ~1:1 with the market factor.")

# --- 2. Model variance vs actual trailing volatility ---
# For each month, compare sqrt(model total variance) with ret_vol from ER
print(f"\n  Model Variance vs Trailing Volatility:")

vol_corrs = []
for month_str, rm in risk_models.items():
    month_period = pd.Period(month_str, "M")
    er_m = er[er["month"] == month_period]

    # Build lookup: permno -> model total variance (monthly)
    model_permnos = rm["permnos"]
    factor_var = np.sum(rm["beta"] ** 2 * np.diag(rm["omega_f"]), axis=1)
    total_var = factor_var + rm["lambda_eps"]
    model_vol = np.sqrt(total_var)  # monthly std

    model_df = pd.DataFrame({
        "permno": model_permnos, "model_vol": model_vol
    })

    merged = er_m[["permno", "ret_vol"]].merge(model_df, on="permno")
    if len(merged) > 30:
        corr = merged["model_vol"].corr(merged["ret_vol"])
        vol_corrs.append(corr)

vol_corrs = np.array(vol_corrs)
print(f"    Avg correlation: {vol_corrs.mean():.4f}")
print(f"    Std:             {vol_corrs.std():.4f}")
print(f"    Min:             {vol_corrs.min():.4f}")
print(f"    Max:             {vol_corrs.max():.4f}")
print(f"    Months:          {len(vol_corrs)}")

if vol_corrs.mean() > 0.3:
    print(f"    -> Good: model captures cross-sectional variance structure")
elif vol_corrs.mean() > 0.1:
    print(f"    -> Acceptable: moderate alignment with realized vol")
else:
    print(f"    -> Warning: low correlation, model may need adjustment")

# --- 3. Positive definiteness check ---
print(f"\n  Positive Definiteness:")
print(f"    Factor Omega diagonal > 0: always True (eigenvalues > 0)")
print(f"    Idio Lambda > 0: checking...")
n_negative_lambda = 0
for rm in risk_models.values():
    if (rm["lambda_eps"] <= 0).any():
        n_negative_lambda += 1
if n_negative_lambda == 0:
    print(f"    All lambda_eps > 0 across all months")
else:
    print(f"    Warning: {n_negative_lambda} months have non-positive lambda_eps")
print(f"    -> Sigma = beta @ Omega @ beta.T + diag(Lambda) is PD")

  VALIDATION

  Beta distributions (month: 2018-06):
  Stocks: 633, PCs: 30
    PC     Mean      Std      Min      Max  |beta|>2
  ----------------------------------------------
  PC 1   0.0024   0.0007  -0.0001   0.0072      0.0%
  PC 2   0.0009   0.0018  -0.0041   0.0080      0.0%
  PC 3  -0.0000   0.0025  -0.0148   0.0046      0.0%
  PC 4   0.0001   0.0019  -0.0061   0.0086      0.0%
  PC 5  -0.0002   0.0020  -0.0100   0.0066      0.0%
  PC 6  -0.0001   0.0018  -0.0114   0.0045      0.0%
  PC 7  -0.0000   0.0016  -0.0086   0.0099      0.0%
  PC 8   0.0001   0.0015  -0.0067   0.0114      0.0%
  PC 9   0.0000   0.0016  -0.0056   0.0122      0.0%
  PC10   0.0001   0.0015  -0.0064   0.0072      0.0%
  PC11  -0.0001   0.0015  -0.0100   0.0094      0.0%
  PC12  -0.0002   0.0014  -0.0070   0.0103      0.0%
  PC13   0.0001   0.0017  -0.0076   0.0060      0.0%
  PC14  -0.0002   0.0020  -0.0077   0.0093      0.0%
  PC15   0.0000   0.0016  -0.0058   0.0159      0.0%
  PC16   0.0001   0.0019  -

In [16]:
# ============================================================
# Cell 8: Quality Control Checks
# ============================================================

print("=" * 60)
print("  TASK 3.1 QUALITY CONTROL CHECKS")
print("=" * 60)

# 1) All months produced a model
expected_months = len(rebal_months)
actual_months = len(risk_models)
assert actual_months >= expected_months * 0.90, \
    f"Too few months: {actual_months}/{expected_months} ({actual_months/expected_months:.0%})"
print(f"1. Models produced: {actual_months}/{expected_months} months "
      f"({actual_months/expected_months:.0%})")

# 2) Coverage per month
min_coverage = diag["pct_coverage"].min()
assert min_coverage > 80, f"Coverage too low: {min_coverage:.1f}%"
print(f"2. Min coverage: {min_coverage:.1f}% (threshold: 80%)")

# 3) PCA variance explained
min_var = diag["var_explained"].min()
assert min_var > 0.70, f"Variance explained too low: {min_var:.1%}"
print(f"3. Min variance explained: {min_var:.1%} (threshold: 70%)")

# 4) Condition number
max_cond = diag["cond_number"].max()
print(f"4. Max condition number (EWMA cov): {max_cond:.0f}")

# 5) Beta range
all_betas = np.concatenate([rm["beta"].ravel() for rm in risk_models.values()])
beta_absmax = np.abs(all_betas).max()
pct_extreme = (np.abs(all_betas) > 5).mean() * 100
print(f"5. Beta range: [{all_betas.min():.3f}, {all_betas.max():.3f}], "
      f"|beta|>5: {pct_extreme:.2f}%")

# 6) Lambda_eps all positive
all_lambdas = np.concatenate([rm["lambda_eps"] for rm in risk_models.values()])
assert (all_lambdas > 0).all(), "Non-positive lambda_eps detected!"
print(f"6. All lambda_eps > 0 (min: {all_lambdas.min():.2e})")

# 7) Factor variance fraction in reasonable range
avg_fvf = diag["factor_var_frac"].mean()
assert 0.2 < avg_fvf < 0.8, f"Factor var fraction out of range: {avg_fvf:.2f}"
print(f"7. Avg factor variance fraction: {avg_fvf:.1%} (expected: 20-80%)")

# 8) Model vol vs trailing vol correlation
assert vol_corrs.mean() > 0.1, \
    f"Low vol correlation: {vol_corrs.mean():.4f}"
print(f"8. Avg model-vs-actual vol correlation: {vol_corrs.mean():.4f}")

# 9) Risk model permnos align with expected returns
n_misalign = 0
for month_str, rm in risk_models.items():
    month_period = pd.Period(month_str, "M")
    er_permnos = set(er.loc[er["month"] == month_period, "permno"])
    model_permnos = set(rm["permnos"])
    # Model should be a subset of ER universe
    extra = model_permnos - er_permnos
    if extra:
        n_misalign += 1
if n_misalign == 0:
    print(f"9. Risk model permnos align with expected returns universe")
else:
    print(f"9. Warning: {n_misalign} months have extra permnos")

# 10) Core stock filtering summary
if "n_core_used" in diag.columns:
    avg_core = diag["n_core"].mean()
    avg_used = diag["n_core_used"].mean()
    print(f"10. Core stocks: avg selected={avg_core:.0f}, "
          f"avg used for cov={avg_used:.0f} "
          f"({avg_used/avg_core:.0%} retained)")

print("\n" + "=" * 60)
print("  ALL CHECKS PASSED")
print("=" * 60)

  TASK 3.1 QUALITY CONTROL CHECKS
1. Models produced: 157/157 months (100%)
2. Min coverage: 99.8% (threshold: 80%)
3. Min variance explained: 83.5% (threshold: 70%)
4. Max condition number (EWMA cov): 1097
5. Beta range: [-0.067, 0.059], |beta|>5: 0.00%
6. All lambda_eps > 0 (min: 6.03e-05)
7. Avg factor variance fraction: 56.9% (expected: 20-80%)
8. Avg model-vs-actual vol correlation: 0.8505
9. Risk model permnos align with expected returns universe
10. Core stocks: avg selected=55, avg used for cov=50 (91% retained)

  ALL CHECKS PASSED


In [17]:
# ============================================================
# Cell 9: Save Outputs
# ============================================================

# --- 1. Risk model pickle ---
pkl_path = DATA_DIR + "sp500_risk_model.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(risk_models, f, protocol=4)
pkl_size = os.path.getsize(pkl_path) / 1e6

# --- 2. Diagnostics CSV ---
diag_path = DATA_DIR + "sp500_risk_model_diagnostics.csv"
diag_out = diag.drop(columns=["year"], errors="ignore").copy()
diag_out["month"] = diag_out["month"].astype(str)
diag_out.to_csv(diag_path, index=False)
diag_size = os.path.getsize(diag_path) / 1e6

# --- Summary ---
print("=" * 70)
print("  OUTPUTS SAVED")
print("=" * 70)

print(f"\n  1) {pkl_path} ({pkl_size:.2f} MB)")
print(f"     Months: {len(risk_models)}")
print(f"     Structure per month:")
print(f"       beta        — (N, K) factor loadings")
print(f"       omega_f     — (K, K) factor covariance (monthly freq)")
print(f"       lambda_eps  — (N,) idiosyncratic variance (monthly freq)")
print(f"       permnos     — stock identifiers (defines row ordering)")
print(f"       n_pcs, cum_var_explained, core_permnos, cond_number, ...")

print(f"\n  2) {diag_path} ({diag_size:.2f} MB)")
print(f"     Monthly diagnostics: coverage, PCA, variance decomposition")

# Show how optimizer will use this
print(f"\n  Usage in Task 3.2 optimizer:")
print(f"    rm = risk_models['2020-06']")
print(f"    beta = rm['beta']       # (N, K)")
print(f"    omega = rm['omega_f']   # (K, K)")
print(f"    lam = rm['lambda_eps']  # (N,)")
print(f"    # Portfolio variance:")
print(f"    # w'Sigma w = (beta.T @ w)' @ omega @ (beta.T @ w)")
print(f"    #            + sum(lam * w**2)")

print(f"\n  Task 3.1 COMPLETE")
print(f"  -> Risk model ready for Task 3.2 (cvxpy optimizer)")

  OUTPUTS SAVED

  1) data/sp500_risk_model.pkl (26.05 MB)
     Months: 157
     Structure per month:
       beta        — (N, K) factor loadings
       omega_f     — (K, K) factor covariance (monthly freq)
       lambda_eps  — (N,) idiosyncratic variance (monthly freq)
       permnos     — stock identifiers (defines row ordering)
       n_pcs, cum_var_explained, core_permnos, cond_number, ...

  2) data/sp500_risk_model_diagnostics.csv (0.02 MB)
     Monthly diagnostics: coverage, PCA, variance decomposition

  Usage in Task 3.2 optimizer:
    rm = risk_models['2020-06']
    beta = rm['beta']       # (N, K)
    omega = rm['omega_f']   # (K, K)
    lam = rm['lambda_eps']  # (N,)
    # Portfolio variance:
    # w'Sigma w = (beta.T @ w)' @ omega @ (beta.T @ w)
    #            + sum(lam * w**2)

  Task 3.1 COMPLETE
  -> Risk model ready for Task 3.2 (cvxpy optimizer)
